# QDM Processing

Python pipeline for processing Quantum Diamond Microscope (QDM) magnetic field
maps: raw B111 map -> cleaned map -> Bz/Bx/By/Bt maps -> figures -> CSV.

**What this notebook does, in order:**
1. **Background removal** (optional) - subtract a blank scan or a constant
   offset from the raw B111 map.
2. **Hot-pixel filtering** (optional) - replace bad/outlier pixels.
3. **Processing** - converts the cleaned B111 map into Bz, Bx, By, Bt maps
   (crop, background subtraction, upward continuation, interpolation as
   needed), and saves the result as a `.mat` file.
4. **Plotting** - Bx/By/Bz/Bt/Bu maps and the LED optical image, side by side.
5. **CSV export** - reads the processed `.mat` file back in and saves it as
   a CSV table, ready for further analysis.

**Input:** a raw `.mat` file containing `B111ferro` and/or `B111para`, plus
an optical/alignment image (`snapshot.*` or `LED.*`) in the same folder.

**Output:** a processed `.mat` file (Bz, Bx, By, Bt, Bu, plus metadata),
summary figures (`.png`), and a CSV table.

**Reference:** Volk, Fu, Trubko, Kehayias, Glenn, Lima (2022), "QDMlab: A
MATLAB toolbox for analyzing quantum diamond microscope (QDM) magnetic
field maps," *Computers & Geosciences* 167, 105198,
https://doi.org/10.1016/j.cageo.2022.105198. Background/field-conversion
methods after Fu, Lima, Volk, Trubko (2020), *G^3*. QDMlab source (open
source, MIT Paleomagnetism Lab / Harvard Paleomagnetics Lab):
https://github.com/HarvardPaleomag/QDMlab

Translated from MATLAB to Python by Ahmed Hussein.

### 1-Processing Functions

In [ ]:
"""
QDM Processing (Curtin) - Python argument-based version
------------------------------------------------------

This is a , notebook pipeline that mirrors the MATLAB QDM Processing workflow,
but without interactive cropping / key-press UI.

all “inputs” pass as function arguments

Main run point:
    QDMdataprocessingCurtin(fullfilename, ...)

Outputs:
    - Returns (Bz, Bx, By, Bt, Bu_uc, h_out, step, corners, newLED)
    - saves a .mat file next to the input

Notes:
    - B111 maps in the MAT file are assumed to be in Gauss (MATLAB multiplies by 1e-4 to get Tesla). # make sure about our unit!!!
"""

import os
import sys
import glob
import numpy as np
import inspect
from datetime import datetime


# ============================================================
# Logging 
# ============================================================
logLevel = "INFO"  # allowed: ALL, DEBUG, INFO, WARN, ERROR, NONE

def logMsg(level, message, lineEnd, indent, returnOnly=False):
    """
    Lightweight logger, close to the MATLAB behavior.
    """
    levels = {'ALL': -1, 'DEBUG': 0, 'INFO': 1, 'WARN': 2, 'ERROR': 3, 'NONE': 4}

    try:
        logLevel_local = globals()['logLevel']
    except Exception:
        logLevel_local = 'INFO'

    try:
        levelIdx = levels[str(level).upper()]
    except Exception:
        levelIdx = 4

    logLevelIdx = levels[str(logLevel_local).upper()]

    if logLevelIdx > levelIdx:
        return None

    try:
        callerName = inspect.stack()[1].function
    except Exception:
        callerName = 'none'

    indent_str = '  ' * int(indent)

    now = datetime.now()
    time_str = now.strftime('%H:%M:%S:') + f"{int(now.microsecond/10000):02d}"

    lvl = str(level).upper()
    lvl_padded = lvl.rjust(7)

    msg = f"<> {time_str:>6} {lvl_padded}: QDMlab.{callerName}:{indent_str} {message}"

    if not returnOnly:
        to_stderr = lvl.upper() in {'INPUT', 'ERROR', 'DEPRECATED', 'ATTENTION'}
        stream = sys.stderr if to_stderr else sys.stdout

        if lineEnd:
            print(msg, file=stream)
        else:
            stream.write(msg)
            stream.flush()
    elif returnOnly and lineEnd:
        msg = msg + "\n"

    return msg


# ============================================================
# Reference banner (prints once per session)
# Prints citation info once.
# ============================================================
_messageDisplayed = {}

def show_references():
    """
    Prints the reference/copyright banner once.
    """
    global _messageDisplayed

    callerName = "QDMdataprocessingCurtin"
    if callerName not in _messageDisplayed:
        _messageDisplayed[callerName] = False

    banner = (
        "QDMdataprocessing_Curtin_Version -- cropping, background compensation, interpolation,\n"
        "  calibration, and upward continuation of QDM data.\n"
        "Reference: Fu, Lima, Volk, Trubko (2020) G^3.\n"
        "--------------------------------------------------------------------------------\n"
        "Matlab code translated by Ahmed Hussein from Matlab to python after Eduardo A. Lima and Roger R. Fu\n"
        "Copyright (C) 2017-2020 MIT Paleomagnetism Lab, Harvard Paleomagnetics Lab\n"
        "--------------------------------------------------------------------------------\n"
    )

    if not _messageDisplayed[callerName]:
        print(banner, end="")
        _messageDisplayed[callerName] = True


# ============================================================
# LED loader (LED.* in folder, or up to 2 levels above)
# LED = optical image (not magnetic)
# Used only for visualization.
# ============================================================
def get_led(path, data=False):
    """
    Looks for LED.* in 'path'. If not found, tries parent and grandparent.
    Loads first match:
      - LED.csv: numeric grid (comma-separated)
      - LED.jpg: image
    """
    if data is not False:
        logMsg('error', "NOT IMPLEMENTED", 1, 0)

    ledFiles = []
    n = 0
    path_ = path

    while len(ledFiles) == 0:
        if n == 2:
            raise RuntimeError(f'NO LED.* files found in << {path} >> or the two directories above!')

        ledFiles = (glob.glob(os.path.join(path_, "snapshot.*")) + glob.glob(os.path.join(path_, "LED.*")))

        if len(ledFiles) == 0:
            logMsg('error', "NO LED.* files found! Going up one directory", 1, 0)
            path_ = os.path.dirname(path_)
            n += 1

    logMsg('debug', f'found {len(ledFiles)} LED.* files', 1, 0)

    led_first_full = ledFiles[0]
    led_first = os.path.basename(led_first_full)
    _, ext = os.path.splitext(led_first)

    # keep the MATLAB-style log message
    #logMsg('debug', f'loading: {os.path.join(path, led_first)}', 1, 0)
    logMsg('debug', f'loading: {led_first_full}', 1, 0)

    ext = ext.lower()
    if ext == '.csv':
        #  CSV is comma-separated
       # return np.loadtxt(os.path.join(path, led_first), delimiter=',')
        return np.loadtxt(led_first_full, delimiter=',')

    if ext == '.jpg':
        from imageio.v2 import imread
        #return imread(os.path.join(path, led_first))
        return imread(led_first_full)

    return None


# ============================================================
# Quadratic background subtraction (poly22)
# ============================================================
# Removes smooth background trends.
#
# Model:
# B(x,y) = c0 + c1x + c2y + c3x² + c4xy + c5y²
#
# Then subtract fitted surface


def QuadBGsub(imIn):
    """
    Fits: c0 + c1*x + c2*y + c3*x^2 + c4*x*y + c5*y^2
    Subtracts fitted surface from imIn.
    """
    imIn = np.asarray(imIn, dtype=float)

    x = np.arange(1, imIn.shape[1] + 1)
    y = np.arange(1, imIn.shape[0] + 1)
    X, Y = np.meshgrid(x, y)

    xData = X.ravel()
    yData = Y.ravel()
    zData = imIn.ravel()

    A = np.column_stack([
        np.ones_like(xData),
        xData,
        yData,
        xData * xData,
        xData * yData,
        yData * yData
    ])

    cvals, _, _, _ = np.linalg.lstsq(A, zData, rcond=None)

    fitSurface = (
        cvals[0]
        + cvals[1] * X
        + cvals[2] * Y
        + cvals[3] * X * X
        + cvals[4] * X * Y
        + cvals[5] * Y * Y
    )

    return imIn - fitSurface


# ============================================================
# Upward continuation in frequency domain
# ============================================================
# Physics:
# Magnetic field decays exponentially with height.
#
# Fourier domain formula:
#
#     B(k,z+dh) = B(k,z) e^{-k dh}
#
# where k = √(kx² + ky²)

def UpCont(Bin, dh, fs):
    """
    Upward continuation: FFT(B) * exp(-k*dh)
    Matches your MATLAB implementation (EXPAND + oversampling).
    """
    EXPAND = 1
    FOVERSAMPL = 2

    if EXPAND:
        Bino = Bin
        Z = np.zeros_like(Bin)
        Bin = np.block([[Z, Z, Z],
                        [Z, Bin, Z],
                        [Z, Z, Z]])

    SIZEx, SIZEy = Bin.T.shape
    Nx = SIZEx * FOVERSAMPL
    Ny = SIZEy * FOVERSAMPL

    f2 = np.concatenate((np.arange(0, Ny // 2 + 1),
                         np.arange(-(Ny // 2 - 1), 0))) * fs / Ny
    f1 = np.concatenate((np.arange(0, Nx // 2 + 1),
                         np.arange(-(Nx // 2 - 1), 0))) * fs / Nx

    F2, F1 = np.meshgrid(f2, f1)

    kx = 2 * np.pi * F1
    ky = 2 * np.pi * F2
    k = np.sqrt(kx**2 + ky**2)

    et = np.exp(-dh * k)

    e = np.fft.fft2(Bin.T, s=(Nx, Ny))
    Bout = np.real(np.fft.ifft2(e * et, s=(Nx, Ny))).T

    Bout = Bout[:Ny // FOVERSAMPL, :Nx // FOVERSAMPL]

    if EXPAND:
        Bout = Bout[Bino.shape[0]:2 * Bino.shape[0],
                    Bino.shape[1]:2 * Bino.shape[1]]

    return Bout


# ============================================================
# Bz from Bu (NV projection inversion)
# ============================================================
# NV measures projection:
#
#   Bu = ux Bx + uy By + uz Bz
#
# Using Maxwell relations in Fourier domain,
# we solve for Bz.

def QDMBzFromBu(Bu, fs, u):
    """
    Retrieves Bz from a measured Bu map (frequency-domain filter).
    """
    N1, N2 = Bu.shape

    f1 = np.concatenate((np.arange(0, N1 // 2),
                         np.arange(-(N1 // 2), 0))) * fs / N1
    f2 = np.concatenate((np.arange(0, N2 // 2),
                         np.arange(-(N2 // 2), 0))) * fs / N2

    F2, F1 = np.meshgrid(f2 + 1e-30, f1 + 1e-30)

    ky = 2 * np.pi * F1
    kx = 2 * np.pi * F2
    k = np.sqrt(kx**2 + ky**2)

    #etz = k / (u[2] * k - u[1] * 1j * ky - u[0] * 1j * kx)
    #den = (u[2]*k - u[1]*1j*ky - u[0]*1j*kx)
    den = (u[2]*k + u[1]*1j*ky - u[0]*1j*kx)   # flip sign of the ky term only to convert by origin bottom
    den[0,0] = np.inf # to avoid 0/0 at DC
    etz = k / den

    e = np.fft.fft2(Bu, s=(N1, N2))
    return np.real(np.fft.ifft2(e * etz, s=(N1, N2)))


# ============================================================
# Bx, By from Bz
# ============================================================
# From ∇·B = 0:
#
#   kx Bx + ky By + kz Bz = 0
#
# Leads to:
#
#   Bx = -i kx/k Bz
#   By = -i ky/k Bz

def MITBxByFromBz(Bz, fs):
    """
    Retrieves Bx and By from a Bz map (frequency-domain relations).
    """
    N1, N2 = Bz.shape

    f1 = np.concatenate((np.arange(0, N1 // 2 + 1),
                         np.arange(-(N1 // 2 - 1), 0))) * fs / N1
    f2 = np.concatenate((np.arange(0, N2 // 2 + 1),
                         np.arange(-(N2 // 2 - 1), 0))) * fs / N2

    F2, F1 = np.meshgrid(f2 + 1e-30, f1 + 1e-30)

    kx = 2 * np.pi * F2
    ky = 2 * np.pi * F1 # bx by was flipped
    k = np.sqrt(kx**2 + ky**2)


    k[0,0] = np.inf # to prevent division by zero at DC
    etx = -1j * kx / k
    ety = -1j * ky / k
    
    e = np.fft.fft2(Bz, s=(N1, N2))
    Bx = np.real(np.fft.ifft2(e * etx, s=(N1, N2)))
    By = np.real(np.fft.ifft2(e * ety, s=(N1, N2)))

    By = -By # converting from qdm origin top to physics origin bottom

    return Bx, By


# ============================================================
# Grid interpolation (matches MATLAB's IFACTOR / interp2 step)
# ============================================================
# MATLAB builds xx/yy over 0..N-1 (original grid) and xxi/xyi over
# 0..N-1 with N*IFACTOR points (same physical extent, finer sampling),
# then does linear interp2. This reproduces that exactly.

from scipy.interpolate import RegularGridInterpolator

def interp2_like_matlab(arr, ifactor):
    """
    Upsample a 2D array by `ifactor` along each axis using linear
    interpolation over the same physical extent (0..N-1), matching
    MATLAB's interp2(...,'linear') usage in QDMdataprocessing.m.
    ifactor=1 returns the array unchanged.
    """
    ifactor = int(ifactor)
    if ifactor == 1:
        return arr

    ny, nx = arr.shape
    yy = np.linspace(0, ny - 1, ny)
    xx = np.linspace(0, nx - 1, nx)
    yyi = np.linspace(0, ny - 1, ny * ifactor)
    xxi = np.linspace(0, nx - 1, nx * ifactor)

    interpolator = RegularGridInterpolator((yy, xx), arr, method="linear")
    YYi, XXi = np.meshgrid(yyi, xxi, indexing="ij")
    pts = np.column_stack([YYi.ravel(), XXi.ravel()])

    return interpolator(pts).reshape(ny * ifactor, nx * ifactor)

# ============================================================
# Background offset check - Fu et al. (2020): background subtraction
# should be accurate to within ~2% of peak signal. Averages MULTIPLE
# quiet patches (e.g. top-left + bottom-right corners).
# ============================================================
def check_background_offset(field, patches, label="field"):
    field = np.asarray(field, dtype=np.float64)
    peak_signal = np.nanmax(np.abs(field))

    patch_means = []
    for i, (r, c) in enumerate(patches):
        patch = field[r, c]
        m = np.nanmean(patch)
        s = np.nanstd(patch)
        patch_means.append(m)
        print(f"[{label}] Patch {i+1}: mean={m:.3e}, std={s:.3e}, n={patch.size}")

    dc_offset = np.nanmean(patch_means)
    pct_of_peak = abs(dc_offset) / peak_signal * 100 if peak_signal > 0 else np.nan

    print(f"[{label}] Peak |signal|:            {peak_signal:.3e}")
    print(f"[{label}] Combined background est.: {dc_offset:.3e}")
    print(f"[{label}] Offset as % of peak:      {pct_of_peak:.2f}%  "
          f"({'OK, within Fu et al. 2% guideline' if pct_of_peak < 2 else 'WARNING: exceeds Fu et al. 2% guideline'})")

    return dc_offset, pct_of_peak
    

# ============================================================
# MAIN PIPELINE (argument-based)
# ============================================================
#def QDMdataprocessingCurtin(
#    fullfilename,
#    *,
#    ferroorpara="ferro",
#    xstep_um=4.70,
#    h_um=5.0,
#    calib=1.0,
 #   alpha_deg=0.0,
 #   beta_deg=0.0,
 #   use_quad_bkg=False,
#   use_const_bkg=False,
  #  updist_um=0.0,
 #   ifactor=1,
#   medfilter=False,
 #   nan_policy="zero",
 #   save=True,
 #   out_prefix=None
#):

def QDMdataprocessingCurtin(
    fullfilename,
    *,
    ferroorpara,
    xstep_um,
    h_um,
    calib,
    alpha_deg,
    beta_deg,
    use_quad_bkg=False,
    use_const_bkg=False,
    updist_um=0.0,
    ifactor=1,
    medfilter=False,
    nan_policy="zero",
    save=True,
    out_prefix=None,
    crop_left=0,
    crop_right=0,
    crop_top=0,
    crop_bottom=0,
    user_const_value_uT=0.0
):

    """
    Process one QDM dataset (MAT file) using arguments (no interactive UI).

    Parameters ( “inputs”)
    --------------------------
    fullfilename : str
        Full path to .mat file containing B111ferro and/or B111para
    ferroorpara : {"ferro","para"}
        Select which field map to use
    xstep_um : float
        Pixel size in micrometers
    h_um : float
        NV-sample distance in micrometers
    calib : float
        Field calibration factor (CAL)
    alpha_deg : float
        Rotation of diamond lattice axes around z-axis (degrees)
    beta_deg : float
        Rotation of the image axes around z-axis (degrees). Applied using bilinear rotation.
    use_quad_bkg : bool
        If True, apply quadratic (sloped-surface) background subtraction.
        Independent of use_const_bkg -- both, either, or neither can be True.
    use_const_bkg : bool
        If True, apply automatic constant (flat DC) background subtraction
        (estimated from quiet corner patches). Independent of use_quad_bkg.
    updist_um : float
        Upward continuation distance in micrometers (0 = none)
    ifactor : int
        Interpolation factor. ifactor=1 keeps the original grid (no interpolation).
        ifactor>1 upsamples Bz, Bu, Bx, By onto a finer grid (linear interp2,
        matches MATLAB QDMdataprocessing.m) and reduces `step` accordingly.
    medfilter : bool
        Apply 4x4 median filter (matches MATLAB option)
    nan_policy : {"zero","keep"}
        "zero" replaces NaNs with 0, "keep" leaves them
    save : bool
        Save .mat output next to input file
    out_prefix : str or None
        Output filename prefix; if None, uses "Bz_uc{updist_um}"
    user_const_value_uT : float
        User-defined constant value to subtract, in microtesla (uT).
        Default 0.0 = no subtraction (off). Fully independent of
        use_quad_bkg and use_const_bkg -- set to any nonzero value to
        subtract exactly that amount.

    Returns
    -------
    Bz, Bx, By, Bt, Bu, h, step, corners, newLED
    """
    show_references()

    # Unit conversions (MATLAB works in meters internally here)
    xstep = float(xstep_um) * 1e-6
    h = float(h_um) #* 1e-6
    updist = float(updist_um) * 1e-6

    # Load MATLAB .mat
    from scipy.io import loadmat, savemat
    mat = loadmat(fullfilename)

    pathname = os.path.dirname(fullfilename)
    savefolder = pathname
    savefilename = os.path.basename(pathname)

    # Load LED image (full, no cropping)
    ledImg = get_led(pathname)
    newLED = ledImg

    # Select field map and convert to Tesla
    ferroorpara = str(ferroorpara).strip().lower()
    if ferroorpara == "ferro":
        if "B111ferro" not in mat:
            raise KeyError("B111ferro not found in MAT file.")
        B = mat["B111ferro"]
    elif ferroorpara == "para":
        if "B111para" not in mat:
            raise KeyError("B111para not found in MAT file.")
        B = mat["B111para"]
    else:
        raise ValueError('ferroorpara must be "ferro" or "para".')

    B = np.asarray(B, dtype=float) * 1e-6  # Field from Micro to tesla 


    corners_raw = [
        (slice(0, 50), slice(0, 50)),
        (slice(-50, None), slice(-50, None)),
    ]
    check_background_offset(B, corners_raw, label="raw B (input, informational only -- no correction applied here)")

    # rotation
    from scipy.ndimage import rotate

    if float(beta_deg) != 0.0:
     logMsg("info", f"Applying beta rotation: {beta_deg} deg", 1, 0)
    # MATLAB imrotate default expands output ("loose"), bilinear
     B = rotate(B, angle=float(beta_deg), reshape=True, order=1, mode="constant", cval=np.nan)


    # NaN handling
    if np.isnan(B).any():
        if nan_policy == "zero":
            logMsg("info", "replacing NaN values with 0", 1, 0)
            B = np.nan_to_num(B, nan=0.0)
        else:
            logMsg("warn", "NaN values present (nan_policy='keep')", 1, 0)



    # Make dimensions even (matches MATLAB)
    if B.shape[1] % 2 == 1:
        B = B[:, 1:]
        #B = B[:, :-1]
    if B.shape[0] % 2 == 1:
        B = B[1:, :]
        #B = B[:-1, :]
        
    # No cropping: corners represent full image
  #  corners = np.array([[1, 1], [B.shape[0], B.shape[1]]], dtype=int)

    # ============================
    # CROP (remove pixels from edges)
    # ============================
    ny, nx = B.shape
    L = int(crop_left)
    R = int(crop_right)
    T = int(crop_top)
    Btm = int(crop_bottom)

    if (L+R) >= nx or (T+Btm) >= ny:
        raise ValueError("Crop too large: removes all pixels.")

    if (L or R or T or Btm):
        B = B[T:ny-Btm, L:nx-R]

        # crop LED too (if it matches size)
        try:
            if newLED is not None and getattr(newLED, "ndim", 0) >= 2:
                if newLED.shape[0] == ny and newLED.shape[1] == nx:
                    newLED = newLED[T:ny-Btm, L:nx-R]
        except Exception:
            pass

    # corners after crop (1-based like MATLAB)
    corners = np.array([[1, 1], [B.shape[0], B.shape[1]]], dtype=int)


    
    # NV axis unit vector (same choice as the MATLAB script)
    U = np.array([-np.sqrt(2/3), 0.0, -np.sqrt(1/3)], dtype=float)

    # Rotation around z-axis (alpha)
    a = float(alpha_deg)
    Ra = np.array([
        [np.cos(np.deg2rad(a)), -np.sin(np.deg2rad(a)), 0.0],
        [np.sin(np.deg2rad(a)),  np.cos(np.deg2rad(a)), 0.0],
        [0.0,                    0.0,                  1.0]
    ], dtype=float)

    U_hat = (Ra @ U.reshape(3, 1)).reshape(3,)
    U_hat = U_hat / np.linalg.norm(U_hat)

    # Background stage: quadratic subtraction (SWITCH 2), applied here
    # since it needs the full B array (after crop/rotation) to fit the surface.
    Bback = B

    # Calibration factor
    Bu = Bback * float(calib)

    # Upward continuation
    fs = 1.0 / xstep
    if updist > 0:
        Bu_uc = UpCont(Bu, updist, fs)
    else:
        Bu_uc = Bu

    # Convert to Bz
    Bz = QDMBzFromBu(Bu_uc, fs, U_hat)

    # ==============================================================
    # ALL THREE background corrections, applied HERE, on Bz --
    # matching Fu et al. (2020)'s protocol: convert to Bz FIRST,
    # correct AFTER. Applied in order: quad -> const -> user value.
    # Each is fully independent -- run any combination.
    # ==============================================================
    if use_quad_bkg:
        logMsg("info", "[use_quad_bkg=True] Applying quadratic subtraction to Bz", 1, 0)
        Bz = QuadBGsub(Bz)

    if use_const_bkg:
        dc_offset, pct_of_peak = check_background_offset(Bz, corners_raw, label="Bz, auto-const estimate")
        logMsg("info", f"[use_const_bkg=True] Subtracting auto-estimated constant from Bz: "
                        f"{dc_offset:.3e} T ({pct_of_peak:.2f}% of peak)", 1, 0)
        Bz = Bz - dc_offset

    if float(user_const_value_uT) != 0.0:
        user_const_value_T = float(user_const_value_uT) * 1e-6   # uT -> T
        logMsg("info", f"[user_const_value_uT={user_const_value_uT}] Subtracting "
                        f"user-defined constant from Bz: {user_const_value_T:.3e} T", 1, 0)
        Bz = Bz - user_const_value_T

    check_background_offset(Bz, corners_raw, label="Bz, after all corrections")

    # Compute Bx, By from Bz
    Bx, By = MITBxByFromBz(Bz, fs)

    # ------------------------------------------------------------
    # Interpolation (matches MATLAB's IFACTOR / interp2 step).
    # ifactor=1 -> arrays unchanged. ifactor>1 -> Bz, Bu, Bx, By are
    # upsampled onto a finer grid spanning the same physical extent.
    # ------------------------------------------------------------
    ifactor = int(ifactor)
    if ifactor < 1:
        raise ValueError("ifactor must be a positive integer (1 = no interpolation).")

    if ifactor > 1:
        logMsg("info", f"Interpolating maps by factor {ifactor}", 1, 0)
        Bz    = interp2_like_matlab(Bz, ifactor)
        Bu_uc = interp2_like_matlab(Bu_uc, ifactor)
        Bx    = interp2_like_matlab(Bx, ifactor)
        By    = interp2_like_matlab(By, ifactor)

    # Median filter (MATLAB-like target maps)
    from scipy.ndimage import median_filter
    if medfilter:
        logMsg("info", "Applying 4x4 median filter to Bu and Bz (MATLAB-like)", 1, 0)
        Bu_uc = median_filter(Bu_uc, size=4)
        Bz = median_filter(Bz, size=4)
        Bx = median_filter(Bx, size=4)
        By = median_filter(By, size=4)

    # Total field (computed AFTER interpolation/medfilter, matches MATLAB order)
    Bt = np.sqrt(Bx**2 + By**2 + Bz**2)

    # step size reflects the interpolation factor (matches MATLAB: step = xstep/IFACTOR)
    step = xstep / ifactor

    # Updated height after upward continuation (same idea as MATLAB)
    #h_out = h + updist
    h_out = h + float(updist_um)

    # Save
    if out_prefix is None:
        out_prefix = f"Bz_uc{updist_um}"

    if save:
        out_mat = os.path.join(savefolder, f"{out_prefix}.mat")
        savemat(out_mat, {
            "Bz": Bz.astype(float),
            "Bx": Bx.astype(float),
            "By": By.astype(float),
            "Bt": Bt.astype(float),
            "Bu": Bu_uc.astype(float),
            "h": float(h_out),
            "step": float(step),
            "corners": corners,
            "newLED": newLED
        })
        #print(f"\nSaving {savefilename}_{out_prefix}.mat...\n")
        print(f"Saving {out_mat}...")

    #return Bz, Bt, Bu_uc, h_out, step, corners, newLED
    return Bz, Bx, By, Bt, Bu_uc, h_out, step, corners, newLED


In [ ]:
# ============================================================
# Hot-pixel filtering (Python translation of filter_hot_pixels.m)
# ============================================================
# Replaces "hot" pixels - values whose absolute value exceeds `threshold`
# (Gauss), and/or values that are `cutOff` standard deviations away from
# the median (of the data itself, or of a `chi` fit-quality map) - with
# the mean of a surrounding (2*winSize+1) x (2*winSize+1) window.
#
# This is a 1:1 translation of filter_hot_pixels.m, including its exact
# quirks:
#   - the absolute `threshold` pre-filter always sets flagged pixels to
#     NaN first, regardless of `includeHotPixel`
#   - pixels are replaced in row-major order using `filteredData`, which
#     is updated as it goes - so a pixel's replacement window can include
#     values that were *already replaced* earlier in the same pass, just
#     like the MATLAB nested for-loop.
#   - winSize=nan -> hot pixels set to NaN; winSize=0 -> hot pixels set
#     to 0; winSize=N -> hot pixels replaced by the window mean.

def filter_hot_pixels(data, cutOff='none', includeHotPixel=False, chi=0,
                       winSize=3, threshold=5, checkPlot=False):
    """
    Parameters
    ----------
    data : 2D array
        Data matrix to be filtered (e.g. B111ferro/Bz), same units as
        `threshold` (MATLAB default: Gauss).
    cutOff : float or 'none' [default 'none']
        How many standard deviations above the median a pixel has to be
        to get filtered. 'none' disables this stdev-based filtering.
    includeHotPixel : bool [default False]
        If True, the replacement mean includes the hot pixel's own
        (pre-threshold) value. If False, the hot pixel is set to NaN
        before averaging.
    chi : 2D array or 0 [default 0]
        If an array the same shape as `data`, hot pixels are flagged
        using this map (e.g. Lorentzian fit chi^2) instead of `data`.
    winSize : int, 0, or nan [default 3]
        Pixels to each side used for the averaging window. nan -> set
        to NaN. 0 -> set to 0.
    threshold : float or False [default 5]
        Absolute-value threshold (same units as `data`); pixels with
        |data| >= threshold are always flagged.
    checkPlot : bool [default False]
        If True, shows a before / after / delta / hot-pixel-mask plot.

    Returns
    -------
    filteredData : 2D ndarray
        Data with hot pixels replaced.
    filteredPixels : 2D ndarray (0/1)
        Mask of which pixels were flagged/replaced.
    """
    data_in = np.asarray(data, dtype=float)
    data = data_in.copy()
    dshape = data.shape

    chi_is_map = isinstance(chi, np.ndarray) and np.shape(chi) == dshape

    # pre-filter extreme outliers using the absolute threshold (Gauss)
    if threshold is not False and threshold is not None and str(threshold).lower() != 'none':
        toFilter = np.abs(data) >= threshold
    else:
        toFilter = np.zeros(dshape, dtype=bool)

    if threshold and np.count_nonzero(toFilter):
        logMsg('debug', f'found {np.count_nonzero(toFilter)} pixels with field values above {float(threshold):.1e} T', 1, 0)

    if np.any(toFilter):
        data[toFilter] = np.nan

    if cutOff == 'none' and chi_is_map:
        logMsg('error', "filtering according to Chi values, but no 'cutOff' value set. Please set: e.g. 'cutOff', 5", 1, 0)

    dMed = dStd = None

    # stdev-based filtering on data (or on chi, if provided)
    if cutOff != 'none':
        dtype_used = 'chi' if chi_is_map else 'data'
        logMsg('info', f'filtering where {dtype_used} is {cutOff} standard deviations above the median', 1, 0)

        if chi_is_map:
            chi = np.array(chi, dtype=float)
            chiFilter = np.abs(chi) > (np.nanmean(chi) + 15 * np.nanstd(chi))
            chi[chiFilter] = np.nan

            dMed = np.nanmedian(np.abs(chi))
            dStd = np.nanstd(chi)
            logMsg('debug', f'using chi2 values: median = {dMed:.2e}; std = {dStd:.2e}', 1, 0)

            toFilter = toFilter | (chi > dMed + cutOff * dStd) | chiFilter
        else:
            dMed = np.nanmedian(data)
            dStd = np.nanstd(data)
            toFilter = toFilter | (np.abs(data) > dMed + cutOff * dStd)
            logMsg('debug', f'using data values: median = {dMed:.2e}; std = {dStd:.2e}', 1, 0)

    # pixel substitution
    filteredData = data.copy()
    filteredPixels = np.zeros(dshape)

    winSize_is_nan = isinstance(winSize, float) and np.isnan(winSize)

    if winSize_is_nan:
        filteredData[toFilter] = np.nan
        filteredPixels[toFilter] = 1
    elif winSize == 0:
        filteredData[toFilter] = 0
        filteredPixels[toFilter] = 1
    else:
        rows, cols = dshape
        # same row-major order as MATLAB's nested for row / for col loops
        rIdx, cIdx = np.nonzero(toFilter)
        for row, col in zip(rIdx, cIdx):
            filteredPixels[row, col] = 1

            if not includeHotPixel:
                filteredData[row, col] = np.nan

            minrow = max(row - winSize, 0)
            maxrow = min(row + winSize, rows - 1)
            mincol = max(col - winSize, 0)
            maxcol = min(col + winSize, cols - 1)

            window = filteredData[minrow:maxrow + 1, mincol:maxcol + 1]
            filteredData[row, col] = np.nanmean(window)

    n_pixels = int(np.sum(filteredPixels))

    if cutOff == 'none':
        if n_pixels:
            logMsg('info', f'B > +- {float(threshold):.1f}: removed {n_pixels} / {filteredPixels.size} pixel = {n_pixels/filteredPixels.size*100:.2f} percent', 1, 0)
    else:
        if n_pixels:
            logMsg('info', f'filtered by {cutOff} stdev: removed {n_pixels} / {filteredPixels.size} pixel = {n_pixels/filteredPixels.size*100:.2f}% | median = {dMed:.2e}, std = {dStd:.2e}', 1, 0)

    if checkPlot:
        _hot_pixel_checkplot(data_in, filteredData, filteredPixels)

    return filteredData, filteredPixels


def _hot_pixel_checkplot(before, after, filteredPixels):
    """
    Before / after / delta comparison plus a hot-pixel mask map - shown
    when filter_hot_pixels(..., checkPlot=True) is called.
    """
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.2))

    v = np.nanpercentile(np.abs(before), 99)
    if not np.isfinite(v) or v == 0:
        v = np.nanmax(np.abs(before)) or 1.0

    im0 = axes[0].imshow(before, cmap='RdBu_r', vmin=-v, vmax=v, origin='upper')
    axes[0].set_title('before removal')
    plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

    im1 = axes[1].imshow(after, cmap='RdBu_r', vmin=-v, vmax=v, origin='upper')
    axes[1].set_title('after removal')
    plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

    delta = after - before
    vd = np.nanpercentile(np.abs(delta), 99)
    if not np.isfinite(vd) or vd == 0:
        vd = 1.0
    im2 = axes[2].imshow(delta, cmap='RdBu_r', vmin=-vd, vmax=vd, origin='upper')
    axes[2].set_title('delta (after - before)')
    plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

    im3 = axes[3].imshow(filteredPixels > 0, cmap='jet', vmin=-1, vmax=1, origin='upper')
    axes[3].set_title('hot pixels flagged')
    axes[3].axis('off')

    for ax in axes[:3]:
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle('Hot-pixel filtering', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Background removal (Python translation of subtract_blank.m and
# subtract_constant.m from QDMlab)
# ============================================================
# QDMlab reference: Volk, Fu, Trubko, Kehayias, Glenn, Lima (2022),
#   "QDMlab: A MATLAB toolbox for analyzing quantum diamond microscope
#   (QDM) magnetic field maps", Computers & Geosciences 167, 105198.
#   https://doi.org/10.1016/j.cageo.2022.105198
# Background subtraction method: Fu, Lima, Volk, Trubko (2020), G^3.
# QDMlab source: https://github.com/HarvardPaleomag/QDMlab (open source,
#   copyright MIT Paleomagnetism Lab / Harvard Paleomagnetics Lab).
# Translated from MATLAB to Python by Ahmed Hussein.
#
# Two options, matching the two MATLAB functions:
#   subtract_blank_map()    -> Python version of subtract_blank.m
#   subtract_constant_map() -> Python version of subtract_constant.m
# Use ONE of them, not both, on a given raw B111ferro/B111para map.

import cv2


def _detect_field_key(mat):
    """Matches MATLAB's is_B111(): finds which field is present."""
    if "B111ferro" in mat:
        return "B111ferro"
    if "B111para" in mat:
        return "B111para"
    if "Bz" in mat:
        return "Bz"
    raise KeyError("No B111ferro, B111para, or Bz field found in this .mat file.")


def get_image_tform(fixed_img, moving_img, max_features=2000, good_match_pct=0.30):
    """
    Python translation of QDMlab's get_image_tform.m.
    Detects features in both images, matches them, and fits a 2-D affine
    transform that warps `moving_img` onto `fixed_img`'s frame.
    (MATLAB uses detectSURF/ORBFeatures + estimateGeometricTransform2D;
    here we use ORB features, which are free/open-source, + RANSAC.)

    Returns
    -------
    M : 2x3 ndarray
        Affine transform mapping moving_img -> fixed_img.
    n_inliers, n_matches : int
        Diagnostic counts -- more inliers relative to matches means a
        more trustworthy alignment.
    """
    def to_gray_u8(im):
        im = np.asarray(im, dtype=np.float64)
        im = im - np.nanmin(im)
        mx = np.nanmax(im)
        if mx > 0:
            im = im / mx
        im = np.nan_to_num(im, nan=0.0)
        return (im * 255).astype(np.uint8)

    fixed_u8 = to_gray_u8(fixed_img)
    moving_u8 = to_gray_u8(moving_img)

    orb = cv2.ORB_create(max_features)
    kp1, des1 = orb.detectAndCompute(fixed_u8, None)
    kp2, des2 = orb.detectAndCompute(moving_u8, None)

    if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
        raise RuntimeError("Not enough features detected in the alignment images.")

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = sorted(matcher.match(des1, des2), key=lambda m: m.distance)
    n_good = max(4, int(len(matches) * good_match_pct))
    matches = matches[:n_good]

    pts_fixed = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    pts_moving = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)

    M, inliers = cv2.estimateAffine2D(pts_moving, pts_fixed, method=cv2.RANSAC, ransacReprojThreshold=3.0)
    if M is None:
        raise RuntimeError("Affine transform estimation failed -- check the alignment images.")

    return M, int(inliers.sum()), len(matches)


def subtract_blank_map(sample_field, sample_align_img, blank_field, blank_align_img,
                        checkPlot=True, title="blank subtraction",
                        min_inliers=15, min_inlier_ratio=0.5):
    """
    Python translation of subtract_blank.m.
    ...
    min_inliers, min_inlier_ratio : quality thresholds. Below these, the
        alignment is flagged unreliable -- happens when the alignment
        images don't share enough real visual texture (e.g. a near-
        featureless/black blank scan).
    """
    sample_field = np.asarray(sample_field, dtype=np.float64)
    blank_field = np.asarray(blank_field, dtype=np.float64)

    M, n_inliers, n_matches = get_image_tform(sample_align_img, blank_align_img)
    inlier_ratio = n_inliers / n_matches if n_matches else 0.0
    reliable = (n_inliers >= min_inliers) and (inlier_ratio >= min_inlier_ratio)

    logMsg("info", f"blank alignment: {n_inliers}/{n_matches} inlier features ({inlier_ratio:.0%})", 1, 0)
    if not reliable:
        logMsg("warn", "alignment looks UNRELIABLE (too few consistent matches). "
                        "Compare the 'aligned' and 'unaligned' panels below and "
                        "judge which one actually looks cleaner before trusting "
                        "either -- or use subtract_constant_map instead if the "
                        "sample and blank were never physically moved.", 1, 0)

    H, W = sample_field.shape
    blank_warped = cv2.warpAffine(blank_field, M, (W, H), flags=cv2.INTER_LINEAR, borderValue=0.0)
    blank_warped[blank_warped == 0] = np.nan

    corrected = sample_field - blank_warped

    def _safe_std(a):
        # avoids "Degrees of freedom <= 0" warnings when a slice is
        # entirely (or almost entirely) NaN, e.g. after a bad warp
        a = a[np.isfinite(a)]
        return np.std(a) if a.size > 1 else np.nan

    if checkPlot:
        import matplotlib.pyplot as plt
        naive = sample_field - blank_field
        fig, axs = plt.subplots(2, 2, figsize=(10, 8))
        v = np.nanpercentile(np.abs(sample_field), 99) or 1.0

        im00 = axs[0, 0].imshow(sample_field, cmap="RdBu_r", vmin=-v, vmax=v)
        axs[0, 0].set_title("sample (original)")
        fig.colorbar(im00, ax=axs[0, 0], fraction=0.046, pad=0.04)

        im01 = axs[0, 1].imshow(blank_field, cmap="RdBu_r", vmin=-v, vmax=v)
        axs[0, 1].set_title("blank (raw)")
        fig.colorbar(im01, ax=axs[0, 1], fraction=0.046, pad=0.04)

        im10 = axs[1, 0].imshow(naive, cmap="RdBu_r", vmin=-v, vmax=v)
        axs[1, 0].set_title(f"unaligned sub (std={_safe_std(naive):.2e})")
        fig.colorbar(im10, ax=axs[1, 0], fraction=0.046, pad=0.04)

        im11 = axs[1, 1].imshow(corrected, cmap="RdBu_r", vmin=-v, vmax=v)
        tag = "" if reliable else " -- UNRELIABLE"
        axs[1, 1].set_title(f"aligned sub (std={_safe_std(corrected):.2e}){tag}")
        fig.colorbar(im11, ax=axs[1, 1], fraction=0.046, pad=0.04)

        for ax in axs.ravel():
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(title, fontsize=14, fontweight="bold")
        fig.tight_layout()
        plt.show()

    return corrected, {"n_inliers": n_inliers, "n_matches": n_matches, "M": M, "reliable": reliable}

def subtract_blank_simple(sample_field, blank_field, checkPlot=True, title="blank subtraction (no alignment)"):
    """
    Direct pixel-by-pixel subtraction, no image registration. Use this
    when the sample and blank were scanned without moving the diamond
    (so the pixel grids are already aligned), or when feature-based
    alignment keeps failing due to low-texture alignment images.
    """
    sample_field = np.asarray(sample_field, dtype=np.float64)
    blank_field = np.asarray(blank_field, dtype=np.float64)
    if sample_field.shape != blank_field.shape:
        raise ValueError(f"Shape mismatch: sample {sample_field.shape} vs blank {blank_field.shape}")

    corrected = sample_field - blank_field

    if checkPlot:
        import matplotlib.pyplot as plt
        fig, axs = plt.subplots(1, 3, figsize=(13, 4.2))
        v = np.nanpercentile(np.abs(sample_field), 99) or 1.0
        for ax, data, t in zip(axs, [sample_field, blank_field, corrected],
                                ["sample", "blank", f"corrected (std={np.nanstd(corrected):.2e})"]):
            im = ax.imshow(data, cmap="RdBu_r", vmin=-v, vmax=v)
            ax.set_title(t)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        fig.suptitle(title, fontsize=13, fontweight="bold")
        fig.tight_layout()
        plt.show()

    return corrected
    

def subtract_constant_map(data, row, col, checkPlot=True, title="constant subtraction"):
    """
    Python translation of subtract_constant.m.
    Subtracts the value at a single chosen pixel (row, col) from the
    whole map. No alignment needed -- this is for removing a simple
    constant offset, not a spatially-varying blank pattern.

    Parameters
    ----------
    data : 2D array
        The raw B111ferro/B111para/Bz map.
    row, col : int
        Pixel location to pick the constant from (0-indexed). Look at the
        checkPlot marker to confirm you picked a sensible reference point
        (an area with no real magnetic signal), then adjust and re-run if
        needed -- this mirrors MATLAB's interactive point-pick, just done
        by re-running the cell with different row/col instead of clicking.

    Returns
    -------
    corrected : 2D ndarray
        data - data[row, col]
    const : float
        The subtracted value.
    """
    data = np.asarray(data, dtype=np.float64)
    const = float(data[row, col])
    corrected = data - const
    logMsg("info", f"Constant value of {const:.3e} subtracted from map", 1, 0)

    if checkPlot:
        import matplotlib.pyplot as plt
        fig, axs = plt.subplots(1, 2, figsize=(9, 4))
        v = np.nanpercentile(np.abs(data), 99) or 1.0
        axs[0].imshow(data, cmap="RdBu_r", vmin=-v, vmax=v, origin="upper")
        axs[0].plot(col, row, 'k+', ms=14, mew=2)
        axs[0].set_title("before (marker = pick point)")
        axs[1].imshow(corrected, cmap="RdBu_r", vmin=-v, vmax=v, origin="upper")
        axs[1].set_title("after")
        fig.suptitle(f"{title}: subtracted {const:.3e}", fontsize=13, fontweight="bold")
        fig.tight_layout()
        plt.show()

    return corrected, const


### 1a-Background Removal (raw B111ferro/B111para, before hot-pixel filtering)

Removes background signal from the raw map before any further processing.
Choose **ONE** of the two options below, matching QDMlab's `subtract_blank`
and `subtract_constant` (Volk et al. 2022, *Computers & Geosciences* 167,
105198; method after Fu, Lima, Volk, Trubko 2020, *G^3*). Skip this whole
section if you don't need background removal.

- **Blank subtraction**: use if you have a separate background scan (no
  sample, diamond only) -- e.g. `Background.mat`. Aligns the blank to the
  sample using the optical images, then subtracts.
- **Constant subtraction**: use if you just want to remove a simple offset
  by picking one reference pixel on the sample map itself -- no blank scan
  needed.

QDMlab is open-source, MIT Paleomagnetism Lab / Harvard Paleomagnetics Lab.
Translated from MATLAB to Python by Ahmed Hussein.

### 1b-Hot Pixel Filtering (raw B111ferro, before processing)

In [ ]:
# ============================================================
# Hot-pixel filtering - raw B111ferro, BEFORE processing
# ============================================================
# Mirrors the MATLAB workflow (threshold_QDM_map / filter_hot_pixels),
# which filters hot pixels on the RAW B111ferro/B111para map straight
# out of the microscope - before any background subtraction, upward
# continuation, or Bz/Bx/By inversion. Filter first, then point
# QDMdataprocessingCurtin at the filtered file.
#
# `threshold` is in the SAME units as the raw .mat file - no Gauss/
# Tesla conversion happens here, that only happens later inside
# QDMdataprocessingCurtin (B = raw * 1e-6). Check the raw file's
# native units before picking a threshold; 5 is the MATLAB default.

DIR = os.getcwd()
raw_file = os.path.join(DIR, "B111ferro.mat")

from scipy.io import loadmat, savemat

raw = loadmat(raw_file)
field_key = "B111ferro" if "B111ferro" in raw else "B111para"
B111_raw = raw[field_key]

B111_filtered, hotPixelMask = filter_hot_pixels(
    B111_raw,
    threshold=100,        # native units of the raw .mat file (MATLAB default)
    cutOff='none',      # set e.g. cutOff=5 to ALSO filter by stdev-above-median
    includeHotPixel=False,
    winSize=3,          # averaging window = 7x7 pixels
    checkPlot=True,     # -> shows before/after/delta/hot-pixel plot
)

print(f"Flagged / replaced {int(hotPixelMask.sum())} / {hotPixelMask.size} pixels")

# Save the filtered map into a new .mat (mirrors MATLAB's '_thresh' output),
# so QDMdataprocessingCurtin below can be pointed at the filtered file
# instead of the raw one (just change filePath to filtered_file).
filtered_file = raw_file.replace(".mat", "_thresh.mat")
out = dict(raw)
out[field_key] = B111_filtered
savemat(filtered_file, out)
print(f"Saved filtered map to: {filtered_file}")


### 2-Run Processing

In [ ]:
# Folder that contains the QDM .mat file and LED.* image
DIR = os.getcwd()

# ===========================================================
# Run 
# ============================================================
if __name__ == "__main__":
    basePath = DIR
    
    # QDM magnetic map (.mat) from the microscope
    # Must contain B111ferro or B111para
    filePath = os.path.join(basePath, "B111ferro_thresh.mat")

    Bz, Bx, By, Bt, Bu, h, step, corners, led = QDMdataprocessingCurtin(
        filePath,
        ferroorpara="ferro",  # Choose what matches the measurement "ferro" use B111ferro map (REMENANT) #"para" use B111para map (INDUCED)
        xstep_um=2.20, # Pixel size in micrometers (depends on PIXEL binning)
        h_um=5.0, # NV-sample distance (µm)
        calib=1.0,  # Field calibration factor
        alpha_deg=0.0,  # Rotation of NV axis around z (degrees)
        beta_deg=0, # Image rotation
        use_quad_bkg= False,     # True/False -- quadratic background
        use_const_bkg= False,   # True/False -- automatic constant background
        user_const_value_uT=0.320,  # value in uT; 0 = off
        updist_um=5.0, # Upward continuation height (µm)
        ifactor=1, # Interpolation factor. 1 = no interpolation (recommended for inversion work). >1 upsamples Bz/Bu/Bx/By onto a finer grid.
        medfilter=False,     # Apply 4×4 median filter to reduce noise # Use only if data is very noisy
        nan_policy="zero",     # "zero" > replace NaNs with 0 (recommended), # "keep" > leave NaNs (can break FFT)
        save=True,     # True  > saves processed .mat file, # False → no saving
        out_prefix="Bz_ferro", # OUTname .mat 
        crop_left=10,  #PIXELS TO BE REMOVED FROM LEFT 
        crop_right=10,#PIXELS TO BE REMOVED FROM RIGHT 
        crop_top=10, #PIXELS TO BE REMOVED FROM TOP 
        crop_bottom=10 #PIXELS TO BE REMOVED FROM BOTTOM 
    )

    print("Bz shape:", Bz.shape)

    corners_out = [
        (slice(0, 50), slice(0, 50)),
        (slice(-50, None), slice(-50, None)),
    ]
    check_background_offset(Bz, corners_out, label="processed Bz (output)")


### 3-Plot Outputs

In [ ]:
#Headers
from scipy.io import loadmat

out_file = os.path.join(basePath, "Bz_ferro.mat")
data = loadmat(out_file)

print("\nVariables inside output:")
print([k for k in data.keys() if not k.startswith("__")])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
from mpl_toolkits.axes_grid1 import ImageGrid

# ------------------------------------------------------------------
# Style
# ------------------------------------------------------------------
mpl.rcParams.update({
    "font.size": 12,
    "font.family": "sans-serif",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.facecolor": "white",
    "axes.edgecolor": "0.3",
})

pix_led = 1.09   # µm / pixel
pix_mag = 2.2    # µm / pixel (binned)

# shapes
ny_mag, nx_mag = Bx.shape
ny_led, nx_led = led.shape

# physical size (should match if they cover same FOV)
Lx_mag = nx_mag * pix_mag
Ly_mag = ny_mag * pix_mag
Lx_led = nx_led * pix_led
Ly_led = ny_led * pix_led

print("FOV mag (µm):", Lx_mag, Ly_mag)
print("FOV led (µm):", Lx_led, Ly_led)

# choose ONE extent (prefer magnetic FOV)
extent_um = [0, Lx_mag, 0, Ly_mag]

fields = [Bx, By, Bz, Bt, Bu, led]
titles = [r"$B_x$", r"$B_y$", r"$B_z$", r"$B_t$", r"$B_u$", "LED (optical)"]

# ------------------------------------------------------------------
# ImageGrid: keeps equal-aspect images AND matches each colorbar's
# height to its own subplot exactly (regular fig.colorbar can't do this
# once extent + equal aspect shrink the image inside its axes box).
# ------------------------------------------------------------------
fig = plt.figure(figsize=(15, 7.2))
grid = ImageGrid(fig, [0.03, 0.03, 0.94, 0.88],
                  nrows_ncols=(2, 3),
                  axes_pad=(0.65, 0.35),   # (horizontal, vertical) spacing
                  cbar_mode="each",
                  cbar_location="right",
                  cbar_pad=0.10,
                  cbar_size="5%")

for i, (F, t) in enumerate(zip(fields, titles)):
    ax = grid[i]
    cax = grid.cbar_axes[i]

    if t == "LED (optical)":
        im = ax.imshow(F, cmap="gray", origin="upper", extent=extent_um)
        cax.colorbar(im)
    else:
        F_nT = F * 1e9  # Tesla -> nT
        v = np.percentile(np.abs(F_nT), 99)
        if t == r"$B_t$":
            # Bt is non-negative (field magnitude) -> sequential colormap
            im = ax.imshow(F_nT, cmap="inferno", vmin=0, vmax=v,
                            origin="upper", extent=extent_um)
        else:
            im = ax.imshow(F_nT, cmap="RdBu_r", vmin=-v, vmax=v,
                            origin="upper", extent=extent_um)
        cax.colorbar(im)
        cax.set_title("nT", fontsize=10, pad=4)

    ax.set_title(t, pad=6)
    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")
    ax.tick_params(direction="out", length=4)

fig.suptitle("QDM Magnetic Field Maps", fontsize=16, fontweight="bold", y=0.98)

plt.savefig(os.path.join(DIR, "processing_maps2_ferro.png"), dpi=300, bbox_inches="tight")
plt.show()

### 4-Read Outputs and save as csv ready for analysis

In [ ]:
import numpy as np
import scipy.io
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from mpl_toolkits.axes_grid1 import ImageGrid

# ============================================================
# Constants
# ============================================================
METER_TO_MICROMETER = 1e6
TESLA_TO_NANOTESLA = 1e9
FIELD_KEYS = ["Bx", "By", "Bz"]   # components the loader looks for

LONG_NAMES = {
    "bx": "in-plane magnetic field (x)",
    "by": "in-plane magnetic field (y)",
    "bz": "vertical magnetic field",
}


# ============================================================
# Loader - supports selecting which components to load/export.
# Only uses what is actually saved in the .mat file. If a requested
# component isn't there, it's skipped with a warning - no derivation.
# If the file only has Bz and Bx/By are needed, rerun the earlier
# processing step (QDMdataprocessingCurtin) with Bx/By saved.
# ============================================================
def load_qdm_mat(path, components=None, save_csv=True, csv_path=None):
    """
    Load QDM microscopy data from a MATLAB .mat file and return as an
    xarray Dataset. Supports Bx, By, Bz (any combination present).

    Parameters
    ----------
    path : str or Path
        Path to the .mat file.
    components : list of {"Bx","By","Bz"}, optional
        Which field components to load/export. Examples:
            components=["Bz"]          -> Bz only
            components=["Bx","By"]     -> Bx and By only
            components=["Bx","By","Bz"] or None -> everything present
        If a requested component is not present in the file, it is
        skipped with a warning (it is not derived from other components).
    save_csv : bool, optional (default=True)
        If True, save the loaded component(s) as a CSV file.
    csv_path : str or Path, optional
        Path to save the CSV file. If None, saves as "<mat_file_stem>.csv"
        in the same folder as `path`.

    Returns
    -------
    xarray.Dataset
        Dataset with:
          - bx(y, x), by(y, x), bz(y, x): magnetic field components [nT]
            (only the components that were both requested and present
            in the file are included)
          - coords: x [µm], y [µm], z [µm] (sensor-sample distance)
          - attributes: units + file metadata
    """
    path = Path(path)
    contents = scipy.io.loadmat(path)

    spacing = contents["step"].ravel()[0] * METER_TO_MICROMETER

    # Which components were asked for
    requested_keys = FIELD_KEYS if components is None else list(components)
    invalid = [c for c in requested_keys if c not in FIELD_KEYS]
    if invalid:
        raise ValueError(f"Invalid component(s) {invalid}; choose from {FIELD_KEYS}")

    present_keys = [k for k in requested_keys if k in contents]
    missing_keys = [k for k in requested_keys if k not in contents]

    if missing_keys:
        print(f"[WARN] Requested component(s) {missing_keys} not found in "
              f"{path.name}; skipping. If you need these, rerun the "
              f"processing step with Bx/By saved, or request only the "
              f"components that are present ({[k for k in FIELD_KEYS if k in contents]}).")
    if not present_keys:
        raise KeyError(
            f"None of the requested components {requested_keys} found in "
            f"{path.name}. Available keys: {list(contents.keys())}"
        )

    # Load each present/requested component and convert T -> nT
    fields = {k: contents[k] * TESLA_TO_NANOTESLA for k in present_keys}
    ref_field = fields[present_keys[0]]

    sensor_sample_distance = float(contents["h"].ravel()[0])

    # Coordinates
    x = np.arange(ref_field.shape[1]) * spacing
    y = np.arange(ref_field.shape[0])[::-1] * spacing  # image coords -> cartesian
    z = np.full_like(ref_field, sensor_sample_distance)

    data_vars = {k.lower(): (("y", "x"), v) for k, v in fields.items()}
    data = xr.Dataset(
        data_vars=data_vars,
        coords={
            "x": ("x", x, {"units": "µm"}),
            "y": ("y", y, {"units": "µm"}),
            "z": (("y", "x"), z, {"long_name": "sensor sample distance", "units": "µm"}),
        },
        attrs={"file_name": str(path)},
    )

    for k in data_vars:
        data[k].attrs = {"long_name": LONG_NAMES.get(k, k), "units": "nT"}

    # --- Save to CSV if requested ---
    if save_csv:
        if csv_path is None:
            suffix = "_" + "".join(k.lower() for k in present_keys) if components is not None else ""
            csv_path = path.with_name(path.stem + suffix + ".csv")

        df_dict = {
            "x (µm)": np.tile(x, len(y)),
            "y (µm)": np.repeat(y, len(x)),
            "z (µm)": np.repeat(sensor_sample_distance, len(x) * len(y)),
        }
        for k, v in fields.items():
            df_dict[f"{k.lower()} (nT)"] = v.ravel(order="C")

        df = pd.DataFrame(df_dict)
        df = df.sort_values(by=["y (µm)", "x (µm)"], ascending=[True, True])
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Saved CSV file at: {csv_path}")

    return data


# ============================================================
# Multi-panel plot for Bx / By / Bz (1, 2, or 3 panels, whichever
# are present in `data`). Equal-aspect images, colorbar matched to
# each panel's height.
# ============================================================
def plot_qdm_fields(data, components=None, vlim_percentile=99,
                     cmap="seismic", save_path=None, suptitle="QDM Magnetic Field Maps"):
    """
    Plot any combination of bx, by, bz from an xarray Dataset returned by
    load_qdm_mat.

    Parameters
    ----------
    data : xarray.Dataset
        Output of load_qdm_mat.
    components : list of {"bx","by","bz"}, optional
        Which fields (already loaded in `data`) to plot. Default: all
        data variables present in `data`, in Bx, By, Bz order.
    vlim_percentile : float
        Percentile used to set a symmetric ±v color scale (robust to outliers).
    cmap : str
        Diverging colormap for the field maps.
    save_path : str or Path, optional
        If given, the figure is saved to this path (dpi=300).
    suptitle : str
        Figure title.
    """
    mpl.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
    })

    order = ["bx", "by", "bz"]
    avail = [v for v in order if v in data.data_vars]
    keys = avail if components is None else [c.lower() for c in components if c.lower() in avail]
    if not keys:
        raise ValueError(f"No requested components found in dataset. Available: {avail}")

    n = len(keys)
    x = data["x"].values
    y = data["y"].values
    extent_um = [x.min(), x.max(), y.min(), y.max()]

    # size the figure to the data's own aspect ratio so there's no dead
    # space above/below the row (imshow+extent forces equal aspect)
    data_aspect = (y.max() - y.min()) / (x.max() - x.min())
    per_panel_width = 5.0
    fig_width = per_panel_width * n
    fig_height = per_panel_width * data_aspect + 1.1   # headroom for title/labels

    fig = plt.figure(figsize=(fig_width, fig_height))
    grid = ImageGrid(fig, [0.03, 0.03, 0.94, 0.80],
                      nrows_ncols=(1, n),
                      axes_pad=0.65,
                      cbar_mode="each",
                      cbar_location="right",
                      cbar_pad=0.10,
                      cbar_size="5%")

    label_map = {"bx": r"$B_x$", "by": r"$B_y$", "bz": r"$B_z$"}

    for i, k in enumerate(keys):
        ax = grid[i]
        cax = grid.cbar_axes[i]
        F = data[k].values
        v = np.percentile(np.abs(F), vlim_percentile)
        im = ax.imshow(F, cmap=cmap, vmin=-10000, vmax=10000, origin="upper", extent=extent_um)
        cax.colorbar(im)
        cax.set_title("nT", fontsize=10, pad=4)
        ax.set_title(label_map.get(k, k), pad=6)
        ax.set_xlabel("X (µm)")
        ax.set_ylabel("Y (µm)")
        ax.tick_params(direction="out", length=4)

    fig.suptitle(suptitle, fontsize=16, fontweight="bold", y=0.98)

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"[INFO] Saved figure at: {save_path}")

    return fig


# ============================================================
# Overlay: a field component (e.g. Bz) on top of the LED (reflected-
# light) image, semi-transparent, same physical FOV.
# ============================================================
def plot_field_on_led(data, led, component="bz", pix_led=1.09, vlim_percentile=99,
                       cmap="seismic", alpha=0.55, save_path=None, title=None):
    """
    Overlay a field map (default: Bz) on the LED optical image, aligned by
    physical field of view (µm).

    Parameters
    ----------
    data : xarray.Dataset
        Output of load_qdm_mat (must contain `component`).
    led : 2D array
        LED / reflected-light image.
    component : {"bx","by","bz"}
        Which field to overlay.
    pix_led : float
        LED image pixel size (µm/pixel).
    vlim_percentile : float
        Percentile for symmetric ±v color scale of the field.
    cmap : str
        Diverging colormap for the field overlay.
    alpha : float
        Opacity of the field overlay (0 = invisible, 1 = opaque).
    save_path : str or Path, optional
        If given, save the figure (dpi=300).
    title : str, optional
        Panel title; defaults to "<Component> overlaid on LED".
    """
    component = component.lower()
    if component not in data.data_vars:
        raise KeyError(f"'{component}' not found in dataset. Available: {list(data.data_vars)}")

    mpl.rcParams.update({
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
    })

    F = data[component].values
    x = data["x"].values
    y = data["y"].values
    extent_mag = [x.min(), x.max(), y.min(), y.max()]

    ny_led, nx_led = led.shape
    extent_led = [0, nx_led * pix_led, 0, ny_led * pix_led]

    v = np.percentile(np.abs(F), vlim_percentile)
    label_map = {"bx": r"$B_x$", "by": r"$B_y$", "bz": r"$B_z$"}
    if title is None:
        title = f"{label_map.get(component, component)} overlaid on LED"

    fig, ax = plt.subplots(figsize=(8, 6.5))

    ax.imshow(led, cmap="gray", origin="upper", extent=extent_led)
    im = ax.imshow(F, cmap=cmap, vmin=-v, vmax=v, origin="upper",
                    extent=extent_mag, alpha=alpha)

    ax.set_title(title, pad=8)
    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")
    ax.tick_params(direction="out", length=4)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("nT", fontsize=11)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        print(f"[INFO] Saved figure at: {save_path}")

    return fig

In [ ]:
import os

# Load Bx, By, Bz - derives Bx/By from Bz automatically if not saved in the .mat
data = load_qdm_mat(os.path.join(DIR, "Bz_ferro.mat"), components=["Bx", "By", "Bz"])

# ONE LINE - Bx, By, Bz panel figure
plot_qdm_fields(data, save_path=os.path.join(DIR, "bxbybz_maps.png"))

# ONE LINE - Bz overlaid on LED, transparency tunable via alpha (0=invisible, 1=opaque)

plot_field_on_led(data, led, component="bz", alpha=0.5, save_path=os.path.join(DIR, "bz_on_led.png"));
